In [2]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path
from collections import namedtuple
import json

project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg
from visualization  import hist_quickShow

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

# colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kAzure, ROOT.kSpring, ROOT.kPink, ROOT.kMagenta, ROOT.kTeal, ROOT.kYellow, ROOT.kViolet, ROOT.kCyan]
# colors = [ROOT.kAzure+9, ROOT.kOrange+1, ROOT.kRed+1, ROOT.kGray+1, ROOT.kViolet-5, ROOT.kRed-5, ROOT.kOrange+8, ROOT.kYellow-5, ROOT.kGray+2, ROOT.kCyan-8]
colors = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]

In [3]:
N_BINS = 5000 # Number of bins for histograms

acq_nrs = ['09',]
acq_nrs.extend([f'{_}' for _ in range(10, 29)])
acq_nrs.pop(18)

files = sorted([list((project_root/'data/260527').glob(f'time_{nr}*.csv'))[0] for nr in acq_nrs])

freqs_kHz = [0.200, 0.400, 0.800, 1.600, 3.200, 6.400, 12.800, 25.600, 52.000, 4.000, 4.500, 4.800, 5.000, 5.500, 5.800, 5.400, 5.300, 5.200, 5.100]

Handler = namedtuple('Handler', ('freq', 'expo', 'data', 'hist', 'led_evts'))

handlers = []
for file, freq in zip(files, freqs_kHz):
    # print(f'{file=}, {freq=}')
    data = np.loadtxt(file, delimiter=',', skiprows=1, usecols=1, dtype=float)

    file_name = file.stem[5:]
    with open(file.parent/f'{file_name}.json') as f:
        meta = json.load(f)
        expo = meta['acquisition']['exposure_sec']

    with open(file.parent/f'{file_name}.csv') as f:
        led_evts = sum(1 for _ in f) - 1

    hist = ROOT.TH1F(f"freq_{freq}", "", N_BINS, 0, data.max())
    hist.Fill(data)
    hist.GetXaxis().SetTitle('Inter-arrival time (s)')
    hist.GetYaxis().SetTitle('Counts')
    
    handlers.append(Handler(freq=freq, expo=expo, data=data, hist=hist, led_evts=led_evts))

handlers

[Handler(freq=0.2, expo=245.36038756370544, data=array([5.300e-05, 1.600e-05, 1.400e-05, ..., 4.962e-03, 4.971e-03,
        4.962e-03], shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x5dcdb9df2ad0>, led_evts=49750),
 Handler(freq=0.4, expo=123.40474534034729, data=array([1.090e-04, 1.700e-05, 1.400e-05, ..., 2.471e-03, 2.461e-03,
        2.471e-03], shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x5dcdba062170>, led_evts=49875),
 Handler(freq=0.8, expo=61.082777976989746, data=array([5.400e-05, 1.600e-05, 1.400e-05, ..., 1.220e-03, 1.219e-03,
        1.223e-03], shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x5dcdb95e1fc0>, led_evts=49937),
 Handler(freq=1.6, expo=30.61537480354309, data=array([5.10e-05, 1.50e-05, 1.30e-05, ..., 5.89e-04, 5.87e-04, 5.94e-04],
       shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x5dcdb928b260>, led_evts=49974),
 Handler(freq=3.2, expo=15.366611957550049, data=array([1.02e-04, 1.60e-05, 1.40e-05, ..., 2.93e-04, 2.91e-04, 2.88e-04],
       shape=(

In [4]:
cv, lg = hist_quickShow([h.hist for h in handlers])

In [5]:
# Frequency vs exposure plot

freqs = np.array([h.freq for h in handlers])
expos = np.array([h.expo for h in handlers])

expoVsFreq = ROOT.TGraph(len(freqs), freqs, expos)

In [6]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)

expoVsFreq.SetMarkerColor(colors[0])
expoVsFreq.SetMarkerStyle(20)

expoVsFreq.SetTitle("Exposure Vs frequency")
expoVsFreq.GetXaxis().SetTitle("Frequency (kHz)")
expoVsFreq.GetYaxis().SetTitle("Exposure (s)")

expoVsFreq.Draw("AP")

cv.SetLogx()
cv.SetLogy()
cv.Draw()

In [7]:
# fit_eVf = ROOT.TF1('Fit_expoVsFreq_pow', '[0] +[1]*x^[2]', 0, 5.1)
# expoVsFreq.Fit(fit_eVf, 'RS')

In [8]:
# Event detection efficiency
expos_expected = np.array([h.led_evts/(1000*h.freq) for h in handlers])
efficiencies = expos_expected/expos
efficiencies

array([1.01381483, 1.01039469, 1.02191243, 1.02019819, 1.01650986,
       0.50408325, 0.33367811, 0.19508399, 0.09797991, 1.01210906,
       1.01643269, 1.01298979, 1.00528882, 0.5097209 , 0.50963952,
       0.50972381, 0.50972975, 0.7557533 , 1.01275993])

Some efficiencies are larger than 1 because, at low frequencies, there are high-energy events from cosmic rains that migh impinge on the BGO in between LED pulses, so the fraction of events captured is larger than the LED event rate. The number of captured events $N_\text{LED, comp} = N_\text{LED, real} + N_\text{bg}$, where the number of background events is given by $\lambda_\text{bg}\cdot T_\text{acq}$

Background events with energies within the energy window of the detector are indistinguishable from LED events in the inter-arrival times histogram. The compound rate, including LED and background events is

$\eta_\text{comp} = \dfrac{N_\text{LED, real} + \lambda_\text{bg} \cdot T_\text{acq}}{f \cdot T_\text{acq}} = \eta_\text{real} + \frac{\lambda_\text{bg}}{f}$

For $f < f_\text{crit}$, where $\eta_\text{real} = 1$,

$\eta_{\text{comp}} = 1 + \dfrac{\lambda_\text{bg}}{f}$

where $f_\text{crit} = 2.5 \mathrm{kHz}$, the frequency at which there is a discontinuity in the acquisition time

In [9]:
# Correcting for cosmic rays high-energy events
F_CRIT = 2.5 # kHz
mask_low = freqs < F_CRIT
lambda_bg_estimates = (efficiencies[mask_low] - 1) * freqs[mask_low]

lambda_bg = np.mean(lambda_bg_estimates)
lambda_bg_err = np.std(lambda_bg_estimates)

print(f"λ_bg = {lambda_bg:.3f} ± {lambda_bg_err:.3f} Hz")

λ_bg = 0.014 ± 0.012 Hz


# Corrección por fondo en la ventana de energía del LED

## Motivación

La eficiencia calculada como $\eta = N_\text{LED,cap} / (f \cdot T_\text{acq})$ puede superar 1 si eventos de radiación de fondo que caen dentro de la ventana de energía del fotopico del LED son contados como eventos del LED. El modelo completo es:

$$\eta_\text{comp}(f) = \eta_\text{real}(f) + \frac{\lambda_\text{bg}}{f}$$

Para $f < f_\text{crit}$ donde $\eta_\text{real} = 1$, cualquier exceso por encima de 1 es atribuible a fondo:

$$\lambda_\text{bg} = \left(\eta_\text{comp} - 1\right) \cdot f = \text{const.}$$

## Resultado

| Parámetro | Valor |
|---|---|
| $\lambda_\text{bg}$ | $0.014 \pm 0.012$ Hz |
| Significancia | $< 1.2\sigma$ |

El resultado es **compatible con cero**: no hay contaminación de fondo significativa en la ventana de energía del LED.

## Interpretación

La incertidumbre estadística intrínseca de cada punto de eficiencia (con $N \approx 50\,000$ eventos) es:

$$\sigma(\eta) \approx \frac{1}{\sqrt{N}} \approx 0.45\%$$

La señal de fondo estimada a la frecuencia más baja del experimento es:

$$\frac{\lambda_\text{bg}}{f_\text{min}} \approx 0.014\% \ll \sigma(\eta)$$

Los valores $\eta > 1$ observados son por lo tanto **fluctuaciones estadísticas normales**, no un efecto sistemático.

## Conclusión

- No se aplica corrección por fondo.
- Las barras de error en la curva de eficiencia son $\sigma(\eta) \approx \eta / \sqrt{N_\text{LED}}$.
- El fotopico del LED está bien aislado del espectro de fondo, confirmando la correcta selección del umbral de energía.

In [10]:
# Efficiency vs exposure plot
effVsFreq = ROOT.TGraph(len(freqs), freqs, efficiencies)

# Efficiency model based on non-paralyzable behavior
tau_init = 1 / 5.2   # initial estimation of dead-time
f_model = np.logspace(np.log10(freqs.min()), np.log10(freqs.max()), 500)
k_model = np.ceil(tau_init * f_model)           # número de pulsos saltados
eta_model = 1.0 / k_model                       # eficiencia del modelo
T_model   = 50 * k_model / f_model               # tiempo de adquisición modelo - 50 k-eventos

eff_model = ROOT.TGraph(len(f_model), f_model, eta_model)
expoVsFreq_model = ROOT.TGraph(len(f_model), f_model, T_model)

In [11]:
n_evts_real = np.array([h.led_evts for h in handlers])
eta_errs = efficiencies/np.sqrt(n_evts_real)
freq_errs = np.array([0.005 for _ in eta_errs])

eff_graphErrs = ROOT.TGraphErrors(len(freqs), freqs, efficiencies, freq_errs, eta_errs)

In [12]:
fcrit = 1000/198
f_crit_line = ROOT.TLine(fcrit, 0.01 , fcrit, 1.12)
f_crit_line.SetLineColor(ROOT.kBlack)
f_crit_line.SetLineWidth(2)
f_crit_line.SetLineStyle(2)
print(fcrit)

eff1_line = ROOT.TLine(0,1, 52,1)
eff1_line.SetLineColor(ROOT.kBlack)
eff1_line.SetLineWidth(2)
eff1_line.SetLineStyle(2)

5.05050505050505


In [13]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1600, 600)
cv.Divide(2,1)
lg1 = ROOT.TLegend(0.14, 0.21, 0.38, 0.38)
lg2 = ROOT.TLegend(0.54, 0.61, 0.78, 0.78)

eff_graphErrs.SetMarkerColor(colors[0])
eff_graphErrs.SetMarkerStyle(20)

eff_graphErrs.SetTitle("Efficiency Vs frequency")
eff_graphErrs.GetXaxis().SetTitle("Frequency (kHz)")
eff_graphErrs.GetYaxis().SetTitle("Efficiency (s)")

eff_model.SetLineStyle(1)
eff_model.SetLineColor(colors[2])
eff_model.SetLineWidth(2)

expoVsFreq_model.SetLineStyle(1)
expoVsFreq_model.SetLineColor(colors[2])
expoVsFreq_model.SetLineWidth(2)

lg1.AddEntry(eff_graphErrs, "Measured")
lg1.AddEntry(eff_model, "Model")

lg2.AddEntry(expoVsFreq, "Measured")
lg2.AddEntry(expoVsFreq_model, "Model")

cv.cd(1)
eff_graphErrs.Draw("AP")
eff_model.Draw("C sames")
f_crit_line.Draw()
eff1_line.Draw()
lg1.Draw()
cv.cd(1).SetLogx()

cv.cd(2)
expoVsFreq.Draw("AP")
expoVsFreq_model.Draw("C sames")
# f_crit_line.Draw()
lg2.Draw()
cv.cd(2).SetLogx()
cv.cd(2).SetLogy()

cv.Draw()

In [14]:
np.ceil(1/fcrit*11)/11

np.float64(0.2727272727272727)

# Fit of $\tau$

In [15]:
import math

# --- Modelo ---
def eta_model_single(f_kHz, tau_ms):
    """Eficiencia para señal periódica en detector no-paralizable."""
    k = math.ceil(tau_ms * f_kHz)   # pulsos saltados; tau en ms, f en kHz → producto adimensional ✓
    return 1.0 / max(k, 1)

def eta_model_array(freqs_kHz, tau_ms):
    return np.array([eta_model_single(f, tau_ms) for f in freqs_kHz])

# --- Chi² ---
def chi2(tau_ms, freqs, etas, errs):
    pred = eta_model_array(freqs, tau_ms)
    return np.sum(((etas - pred) / errs) ** 2)

In [16]:
# --- Scan ---
freqs_arr = np.array([h.freq for h in handlers])
etas_arr  = efficiencies                             # ya lo tienes
errs_arr  = eta_errs                                 # ya lo tienes

tau_scan  = np.linspace(0.10, 0.40, 3000)            # ms — cubre 100–400 μs
chi2_scan = np.array([chi2(t, freqs_arr, etas_arr, errs_arr) for t in tau_scan])

idx_min   = np.argmin(chi2_scan)
tau_best  = tau_scan[idx_min]
chi2_min  = chi2_scan[idx_min]
ndof      = len(freqs_arr) - 1

print(f"τ        = {tau_best*1e3:.2f} μs")
print(f"χ²/ndof  = {chi2_min:.2f} / {ndof} = {chi2_min/ndof:.3f}")

τ        = 188.73 μs
χ²/ndof  = 5459.09 / 18 = 303.283


In [17]:
# --- Intervalo 1σ: donde Δχ² < 1 ---
mask_1sig = chi2_scan < chi2_min + 1.0
tau_lo = tau_scan[mask_1sig][0]
tau_hi = tau_scan[mask_1sig][-1]

print(f"τ = {tau_best*1e3:.2f} μs  [{tau_lo*1e3:.2f}, {tau_hi*1e3:.2f}] μs  (1σ)")

τ = 188.73 μs  [188.73, 192.23] μs  (1σ)


In [18]:
# deltachi1_line = ROOT.TLine(70, 1, 430, 1)
# deltachi1_line.SetLineStyle(2)
# deltachi1_line.SetLineColor(ROOT.kBlack)

# --- Gráfica del perfil de χ² ---
if ROOT.gROOT.FindObject('cv_chi2'):
    ROOT.gROOT.FindObject('cv_chi2').Close()

cv_chi2 = ROOT.TCanvas('cv_chi2', 'Chi2 scan', 800, 500)

g_chi2 = ROOT.TGraph(len(tau_scan),
                     (tau_scan * 1e3).astype('d'),          # a μs para el eje
                     (chi2_scan - chi2_min).astype('d'))    # Δχ²

g_chi2.SetTitle(';#tau (#mus);#Delta#chi^{2}')
g_chi2.SetLineWidth(2)
g_chi2.SetLineColor(colors[0])
g_chi2.Draw('AL')

# línea en Δχ²=1 → intervalo 1σ
line1 = ROOT.TLine(tau_lo*1e3, 1.0, tau_hi*1e3, 1.0)
line1.SetLineStyle(2); line1.SetLineColor(ROOT.kRed); line1.SetLineWidth(2)
line1.Draw()

# marca el mínimo
line_best = ROOT.TLine(tau_best*1e3, 0.0, tau_best*1e3, 5.0)
line_best.SetLineStyle(3); line_best.SetLineColor(ROOT.kGray+1); line_best.SetLineWidth(2)
line_best.Draw()

cv_chi2.Draw()

In [19]:
# --- Actualizar la curva del modelo con tau_best ---
eta_best   = eta_model_array(f_model, tau_best)
T_best     = np.array([50 * math.ceil(tau_best * f) / f for f in f_model])

eff_model_best  = ROOT.TGraph(len(f_model), f_model.astype('d'), eta_best.astype('d'))
expo_model_best = ROOT.TGraph(len(f_model), f_model.astype('d'), T_best.astype('d'))

eff_model_best.SetLineColor(ROOT.kRed+1)
eff_model_best.SetLineWidth(2)
expo_model_best.SetLineColor(ROOT.kRed+1)
expo_model_best.SetLineWidth(2)

In [20]:
fcrit = 1000/198
f_crit_line = ROOT.TLine(fcrit, 9 , fcrit, 260)
f_crit_line.SetLineColor(ROOT.kBlack)
f_crit_line.SetLineWidth(2)
f_crit_line.SetLineStyle(2)
print(fcrit)

5.05050505050505


In [21]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1600, 600)
cv.Divide(2,1)
lg1 = ROOT.TLegend(0.14, 0.21, 0.38, 0.38)
lg2 = ROOT.TLegend(0.54, 0.61, 0.78, 0.78)

eff_graphErrs.SetMarkerColor(colors[0])
eff_graphErrs.SetMarkerStyle(20)

eff_graphErrs.SetTitle("Efficiency Vs frequency")
eff_graphErrs.GetXaxis().SetTitle("Frequency (kHz)")
eff_graphErrs.GetYaxis().SetTitle("Efficiency (s)")

eff_model_best.SetLineStyle(1)
eff_model_best.SetLineColor(colors[2])
eff_model_best.SetLineWidth(2)

expo_model_best.SetLineStyle(1)
expo_model_best.SetLineColor(colors[2])
expo_model_best.SetLineWidth(2)

lg1.AddEntry(eff_graphErrs, "Measured")
lg1.AddEntry(eff_model_best, "Model")

lg2.AddEntry(expoVsFreq, "Measured")
lg2.AddEntry(expo_model_best, "Model")

cv.cd(1)
eff_graphErrs.Draw("AP")
eff_model_best.Draw("C sames")
# f_crit_line.Draw()
lg1.Draw()
cv.cd(1).SetLogx()

cv.cd(2)
expoVsFreq.Draw("AP")
expo_model_best.Draw("C sames")
f_crit_line.Draw()
lg2.Draw()
cv.cd(2).SetLogx()
cv.cd(2).SetLogy()

cv.Draw()

## Estimation via exposure plateau

In [22]:
N_EVENTS = 50000

print(f"{'f (kHz)':>10}  {'T_acq (s)':>10}  {'k':>4}  {'τ_i (μs)':>10}")
print('-' * 42)

tau_plateau_estimates = []
for h in handlers:
    f   = h.freq           # kHz
    T   = h.expo           # s
    k   = math.ceil(tau_best * f)       # tau_best en ms, f en kHz → adimensional ✓
    tau_i = T / N_EVENTS                # segundos  ← fórmula correcta
    tau_plateau_estimates.append(tau_i)
    print(f"{f:>10.1f}  {T:>10.3f}  {k:>4}  {tau_i*1e6:>10.2f}")

tau_plateau_estimates = np.array(tau_plateau_estimates)

# Solo puntos en el plateau (f > 3/tau_best kHz)
mask_plateau = np.array([h.freq for h in handlers]) > 3 / tau_best

tau_plateau     = np.mean(tau_plateau_estimates[mask_plateau])
tau_plateau_err = np.std(tau_plateau_estimates[mask_plateau])

print(f"\nτ (plateau) = {tau_plateau*1e6:.2f} ± {tau_plateau_err*1e6:.2f} μs")

   f (kHz)   T_acq (s)     k    τ_i (μs)
------------------------------------------
       0.2     245.360     1     4907.21
       0.4     123.405     1     2468.09
       0.8      61.083     1     1221.66
       1.6      30.615     1      612.31
       3.2      15.367     1      307.33
       6.4      15.496     2      309.93
      12.8      11.705     3      234.11
      25.6      10.012     5      200.23
      52.0       9.813    10      196.26
       4.0      12.348     1      246.97
       4.5      10.931     1      218.63
       4.8      10.283     1      205.66
       5.0       9.945     1      198.90
       5.5      17.833     2      356.65
       5.8      16.914     2      338.27
       5.4      18.163     2      363.25
       5.3      18.503     2      370.06
       5.2      12.723     1      254.45
       5.1       9.680     1      193.60

τ (plateau) = 198.25 ± 1.98 μs


In [23]:
hist_quickShow([h.hist for h in handlers[6:9]], legends = [f'LED freq {h.freq} kHz' for h in handlers[6:9]], color_offset=0)

(<cppyy.gbl.TCanvas object at 0x5dcdb896e4a0>,
 <cppyy.gbl.TLegend object at 0x5dcdbbf5d400>)

In [129]:
peak_pos = np.array([298.3, 217.9, 177.6, 171.5])
print(f'mean = {np.mean(peak_pos)},  std = {np.std(peak_pos)}, err = {np.std(peak_pos)/2}')

mean = 216.32500000000002,  std = 50.57491349473571, err = 25.287456747367855


In [107]:
len(handlers)

19

In [126]:
hist_quickShow([handlers[5].hist], legends = [f'LED freq {handlers[5].freq} kHz'], color_offset=0)

(<cppyy.gbl.TCanvas object at 0x63e1a7759e80>,
 <cppyy.gbl.TLegend object at 0x63e1a775a270>)

In [25]:
1/150e-6

6666.666666666667

In [76]:
np.ceil(188.73*5.2)

np.float64(982.0)

In [115]:
1/280

0.0035714285714285713